In [0]:
%pip install mlflow pydantic openpyxl

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/query-planner-demo")

Agent = WorkspaceClient()

ALLOWED_CONSTRAINTS_TEXT = """
Allowed dimensions and fields from the facilities_scored table:

location:
- state
- city
- pincode
- lat
- lon

facility_identity:
- facility_type
- operator_type

clinical_specialty:
- specialties

clinical_procedure:
- procedures

clinical_equipment:
- equipment

clinical_capability:
- capabilities

staffing:
- number_doctors
- raw_notes
- reasoning

availability:
- capabilities
- raw_notes

trust:
- trust_score
- trust_flags
- ai_reviewed

evidence:
- raw_notes
- reasoning
"""

PLANNER_PROMPT = f"""
You are a healthcare query planning agent for Indian medical facility search.

You convert a user request into structured search constraints for a downstream retrieval and ranking system.
The downstream system searches a table called facilities_scored.
You must only use dimensions and fields that exist in that table.

Do not recommend any facility.
Do not invent facts.
Do not add fields that are not listed below.
Do not output markdown.
Respond with JSON only.

{ALLOWED_CONSTRAINTS_TEXT}

Return valid JSON with exactly these keys:
medical_need
constraints
urgency
distance_preference
origin
trust_threshold
reasoning_steps

Rules:
- constraints must be an array of objects
- each constraint object must contain exactly these keys:
  - dimension
  - field
  - operator
  - value
  - priority
- dimension must be one of:
  - location
  - facility_identity
  - clinical_specialty
  - clinical_procedure
  - clinical_equipment
  - clinical_capability
  - staffing
  - availability
  - trust
  - evidence
- field must be one of the allowed fields for its dimension
- operator must be one of:
  - equals
  - contains
  - near
  - greater_or_equal
  - less_or_equal
- priority must be one of:
  - hard
  - soft
  - exclude

Interpretation rules:
- Use hard only when the user clearly requires something.
- Use soft when the user expresses preference, desirability, tendency, or flexibility.
- Use exclude when the user explicitly rejects something.
- Only add location constraints if the user explicitly mentions a place or the upstream system provides one.
- Only use distance_preference when the user explicitly asks for nearest, closest, near, nearby, within X km, or another clear distance-sensitive request.
- If the user does not mention any location or distance intent, leave location constraints empty and set distance_preference fields to null/false.
- If the user mentions a state, city, or pincode explicitly and as a requirement, treat it as hard unless the wording is clearly flexible.
- If the user says "must", "only", "required", "has to", "needs to", "without fail", use hard.
- If the user says "prefer", "preferably", "ideally", "should", "better if possible", use soft.
- If the user says "exclude", "avoid", "not", "must not", "should not", use exclude.

Field-selection guidance:
- Use location for state, city, pincode, coordinates, or geographic intent.
- Use clinical_procedure for specific procedures such as appendectomy, c-section, dialysis, hysterectomy.
- Use clinical_specialty for medical specialties such as anesthesiology, orthopedics, pediatrics, emergency medicine.
- Use clinical_equipment for equipment such as ventilator, CT scanner, oxygen, ICU monitor.
- Use clinical_capability for broader capabilities such as trauma care, emergency surgery, ICU support, NICU support.
- Use availability when the request is about 24/7, always open, round-the-clock, emergency availability, or similar language. Prefer field=raw_notes unless the request clearly maps to capabilities.
- Use staffing when the request refers to doctor count, part-time doctors, visiting doctors, consultants, or staffing model. Prefer field=raw_notes or field=reasoning for staffing model language.
- Use trust when the request refers to confidence, verified quality, suspiciousness, or minimum trust score. For trust_score use numeric operators such as greater_or_equal or less_or_equal.
- Use evidence only when the user explicitly asks for supporting evidence, exact text support, proof, or citations.

Distance and location logic:
- Distinguish between explicit location filtering and optional distance ranking.
- A place like Bihar, Patna, or pincode 800001 should normally become a location constraint.
- Origin is only for distance calculation and may be null.
- Never invent coordinates.
- If nearest/closest is requested but no coordinates are available, set distance_preference.nearest_only=true and origin.lat/origin.lon=null.
- If no distance-sensitive intent exists, set nearest_only=false and max_distance_km=null.
- If no origin is available, the downstream system must still work without distance scoring.

Semantic normalization guidance:
- Treat semantically similar phrases as the same intent when planning constraints.
- Examples:
  - "24/7", "24x7", "247", "open 24 hours", "always open", "round the clock" all indicate continuous availability.
  - "appendectomy" and "appendicectomy" refer to the same procedure intent.
  - "part-time doctors", "visiting doctors", and "visiting consultants" may indicate a similar staffing pattern.
- Prefer the canonical user intent in the value field, not every variant.

Output requirements:
- medical_need should be a concise phrase describing the core medical need.
- trust_threshold must be a number between 0 and 1. If the user gives no trust preference, choose a sensible default such as 0.6.
- urgency must be one of: low, medium, high, unknown.
- distance_preference must be an object with exactly these keys:
  - nearest_only
  - max_distance_km
- origin must be an object with exactly these keys:
  - lat
  - lon
- reasoning_steps must be an array of strings.
- reasoning_steps should contain why constraints were added or excluded.
- reasoning_steps should contain how constraints were classified.
- If a value is unknown, use null.
- If there are no constraints for a category, do not invent any.

Examples:

User: "Find a hospital in Bihar that must be open 24/7 and can do appendectomy"
Output constraint examples:
{{"dimension":"location","field":"state","operator":"equals","value":"Bihar","priority":"hard"}}
{{"dimension":"availability","field":"raw_notes","operator":"contains","value":"24/7","priority":"hard"}}
{{"dimension":"clinical_procedure","field":"procedures","operator":"contains","value":"appendectomy","priority":"hard"}}

User: "Find a facility that can do appendectomy"
Output constraint examples:
{{"dimension":"clinical_procedure","field":"procedures","operator":"contains","value":"appendectomy","priority":"hard"}}
distance_preference example:
{{"nearest_only": false, "max_distance_km": null}}
origin example:
{{"lat": null, "lon": null}}

User: "Find the nearest emergency facility in Patna"
Output constraint examples:
{{"dimension":"location","field":"city","operator":"equals","value":"Patna","priority":"hard"}}
{{"dimension":"clinical_capability","field":"capabilities","operator":"contains","value":"emergency care","priority":"hard"}}
distance_preference example:
{{"nearest_only": true, "max_distance_km": null}}
origin example:
{{"lat": null, "lon": null}}
"""

@mlflow.trace
def plan_query(user_query: str):
    response = Agent.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=PLANNER_PROMPT),
            ChatMessage(role=ChatMessageRole.USER, content=user_query),
        ],
        temperature=0.1,
        max_tokens=900
    )
    return response.choices[0].message.content

user_query = "Find the nearest facility in rural Bihar that can perform an emergency appendectomy and typically leverages parttime doctors. It preferably should be open 24/7"
raw_plan = plan_query(user_query)
print(raw_plan)

In [0]:
import json
import re
from typing import Dict, List, Any, Tuple

import mlflow
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [0]:
@mlflow.trace
def parse_json_safely(raw_text: str) -> Dict[str, Any]:
    if raw_text is None:
        raise ValueError("Planner output is None")

    text = raw_text.strip()

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError("No valid JSON object found in planner output")

In [0]:
ALLOWED_DIMENSIONS = {
    "location",
    "facility_identity",
    "clinical_specialty",
    "clinical_procedure",
    "clinical_equipment",
    "clinical_capability",
    "staffing",
    "availability",
    "trust",
    "evidence",
}

ALLOWED_OPERATORS = {
    "equals",
    "contains",
    "near",
    "greater_or_equal",
    "less_or_equal",
}

ALLOWED_PRIORITIES = {
    "hard",
    "soft",
    "exclude",
}

from typing import Dict, Any

def validate_query_plan(query_plan: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(query_plan, dict):
        query_plan = {}

    query_plan.setdefault("medical_need", "")
    query_plan.setdefault("constraints", [])
    query_plan.setdefault("urgency", "unknown")
    query_plan.setdefault("trust_threshold", 0.6)
    query_plan.setdefault("reasoning_steps", [])

    if not isinstance(query_plan["constraints"], list):
        query_plan["constraints"] = []

    if not isinstance(query_plan["reasoning_steps"], list):
        query_plan["reasoning_steps"] = []

    if query_plan["urgency"] not in {"low", "medium", "high", "unknown"}:
        query_plan["urgency"] = "unknown"

    distance_preference = query_plan.get("distance_preference", {})
    if not isinstance(distance_preference, dict):
        distance_preference = {}

    query_plan["distance_preference"] = {
        "nearest_only": bool(distance_preference.get("nearest_only", False)),
        "max_distance_km": distance_preference.get("max_distance_km", None),
    }

    origin = query_plan.get("origin", {})
    if not isinstance(origin, dict):
        origin = {}

    query_plan["origin"] = {
        "lat": origin.get("lat", None),
        "lon": origin.get("lon", None),
    }

    try:
        query_plan["trust_threshold"] = float(query_plan["trust_threshold"])
    except Exception:
        query_plan["trust_threshold"] = 0.6

    return query_plan

In [0]:
from math import radians, sin, cos, sqrt, atan2


def haversine_km(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]:
        return None
    try:
        lat1, lon1, lat2, lon2 = map(float, [lat1, lon1, lat2, lon2])
    except Exception:
        return None

    r = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)

    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return r * c


def distance_to_score(distance_km):
    if distance_km is None:
        return 0.0
    return 1.0 / (1.0 + distance_km / 25.0)


def get_origin_from_query_plan(query_plan):
    origin = query_plan.get("origin", {}) or {}
    return origin.get("lat"), origin.get("lon")


def should_use_distance(query_plan):
    dist_pref = query_plan.get("distance_preference", {}) or {}
    origin_lat, origin_lon = get_origin_from_query_plan(query_plan)

    nearest_only = bool(dist_pref.get("nearest_only", False))
    max_distance_km = dist_pref.get("max_distance_km", None)
    has_origin = origin_lat is not None and origin_lon is not None

    return {
        "distance_requested": nearest_only or max_distance_km is not None,
        "nearest_only": nearest_only,
        "max_distance_km": max_distance_km,
        "has_origin": has_origin,
        "origin_lat": origin_lat,
        "origin_lon": origin_lon,
    }


def text_contains(value, needle):
    if not value or not needle:
        return False
    return str(needle).strip().lower() in str(value).strip().lower()

In [0]:
@mlflow.trace
def split_constraints(query_plan: Dict[str, Any]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    constraints = query_plan.get("constraints", []) or []

    hard_constraints = [c for c in constraints if c.get("priority") == "hard"]
    soft_constraints = [c for c in constraints if c.get("priority") == "soft"]
    exclude_constraints = [c for c in constraints if c.get("priority") == "exclude"]

    return hard_constraints, soft_constraints, exclude_constraints

In [0]:
import json
import mlflow
from typing import Dict, Any
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


@mlflow.trace
def build_search_df(table_name: str = "facilities_scored") -> DataFrame:
    df = spark.table(table_name)

    return (
        df
        .withColumn("lat", F.col("lat").cast("double"))
        .withColumn("lon", F.col("lon").cast("double"))
        .withColumn("state_text", F.lower(F.coalesce(F.col("state"), F.lit(""))))
        .withColumn("city_text", F.lower(F.coalesce(F.col("city"), F.lit(""))))
        .withColumn("pincode_text", F.coalesce(F.col("pincode").cast("string"), F.lit("")))
        .withColumn("facility_type_text", F.lower(F.coalesce(F.col("facility_type"), F.lit(""))))
        .withColumn("operator_type_text", F.lower(F.coalesce(F.col("operator_type"), F.lit(""))))
        .withColumn("specialties_text", F.lower(F.coalesce(F.concat_ws(" ", F.col("specialties")), F.lit(""))))
        .withColumn("procedures_text", F.lower(F.coalesce(F.concat_ws(" ", F.col("procedures")), F.lit(""))))
        .withColumn("equipment_text", F.lower(F.coalesce(F.concat_ws(" ", F.col("equipment")), F.lit(""))))
        .withColumn("capabilities_text", F.lower(F.coalesce(F.concat_ws(" ", F.col("capabilities")), F.lit(""))))
        .withColumn("trust_flags_text", F.lower(F.coalesce(F.concat_ws(" || ", F.col("trust_flags")), F.lit(""))))
        .withColumn("raw_notes_text", F.lower(F.coalesce(F.col("raw_notes"), F.lit(""))))
        .withColumn("reasoning_text", F.lower(F.coalesce(F.col("reasoning"), F.lit(""))))
        .withColumn(
            "search_text",
            F.concat_ws(
                " ",
                F.col("specialties_text"),
                F.col("procedures_text"),
                F.col("equipment_text"),
                F.col("capabilities_text"),
                F.col("raw_notes_text"),
                F.col("reasoning_text"),
                F.col("trust_flags_text"),
            )
        )
        .withColumn("trust_score_safe", F.coalesce(F.col("trust_score"), F.lit(0.0)))
        .withColumn("ai_reviewed_safe", F.coalesce(F.col("ai_reviewed"), F.lit(False)))
    )

In [0]:
SEMANTIC_SYNONYMS = {
    "24/7": ["24/7", "24x7", "247", "24 hours", "24 hour", "open 24 hours", "always open", "round the clock"],
    "appendectomy": ["appendectomy", "appendicectomy", "appendix removal", "appendix removal surgery", "appendicitis surgery"],
    "parttime doctors": ["part-time doctors", "part time doctors", "parttime doctors", "visiting doctor", "visiting doctors", "visiting consultant", "visiting consultants"],
    "icu": ["icu", "intensive care", "critical care", "nicu", "picu"],
}

def normalize_text(x: Any) -> str:
    if x is None:
        return ""
    return str(x).strip().lower()

def expand_semantic_values(value: Any) -> List[str]:
    base = normalize_text(value)
    if not base:
        return []
    return list(dict.fromkeys([base] + SEMANTIC_SYNONYMS.get(base, [])))

In [0]:
def _broad_expr(field: str, operator: str, value: Any):
    v = normalize_text(value)

    field_map = {
        "state": F.col("state_text"),
        "city": F.col("city_text"),
        "pincode": F.col("pincode_text"),
        "facility_type": F.col("facility_type_text"),
        "operator_type": F.col("operator_type_text"),
        "specialties": F.col("specialties_text"),
        "procedures": F.col("procedures_text"),
        "equipment": F.col("equipment_text"),
        "capabilities": F.col("capabilities_text"),
        "raw_notes": F.col("raw_notes_text"),
        "reasoning": F.col("reasoning_text"),
        "trust_flags": F.col("trust_flags_text"),
        "trust_score": F.col("trust_score_safe"),
    }

    col_expr = field_map[field]

    if operator == "equals":
        return col_expr == v

    if operator == "contains":
        variants = expand_semantic_values(v)
        expr = None
        for term in variants:
            current = col_expr.contains(term)
            expr = current if expr is None else (expr | current)
        return expr

    if operator == "greater_or_equal":
        return col_expr >= float(value)

    if operator == "less_or_equal":
        return col_expr <= float(value)

    return None

In [0]:
@mlflow.trace
def retrieve_candidate_pool(query_plan: Dict[str, Any], table_name: str = "facilities_scored", max_candidates: int = 40):
    hard_constraints, soft_constraints, exclude_constraints = split_constraints(query_plan)

    df = build_search_df(table_name)

    for c in hard_constraints:
        if c.get("operator") == "near":
            continue
    
        if c.get("field") in ["state", "city", "pincode", "trust_score", "facility_type", "operator_type"]:
            expr = _broad_expr(c["field"], c["operator"], c["value"])
            if expr is not None:
                df = df.filter(expr)


    for c in exclude_constraints:
        if c.get("operator") == "near":
            continue

        expr = _broad_expr(c["field"], c["operator"], c["value"])
        if expr is not None:
            df = df.filter(~expr)

    text_terms = []
    for c in hard_constraints + soft_constraints:
        if c["field"] in {"specialties", "procedures", "equipment", "capabilities", "raw_notes", "reasoning"}:
            text_terms.extend(expand_semantic_values(c["value"]))

    if text_terms:
        expr = None
        for term in set(text_terms):
            current = F.col("search_text").contains(term)
            expr = current if expr is None else (expr | current)
        df = df.filter(expr)

    rows = [r.asDict(recursive=True) for r in df.limit(max_candidates).collect()]
    return rows

In [0]:
def parse_json_from_llm_text(content):
    if content is None:
        raise ValueError("LLM returned None")

    if not isinstance(content, str):
        return content

    text = content.strip()
    if not text:
        raise ValueError("LLM returned empty content")

    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise ValueError(f"No JSON object found in LLM output: {text[:300]}")

In [0]:
@mlflow.trace
def llm_judge_candidate(user_query: str, query_plan: dict, candidate: dict) -> dict:
    error_result = {
        "match_score": 0.0,
        "trust_penalty": 0.5,
        "should_recommend": False,
        "matched_requirements": [],
        "conflicting_flags": ["LLM_JUDGE_FAILED"],
        "reasoning": "Judge failed to parse response"
    }

    try:
        distance_info = should_use_distance(query_plan)

        facility_payload = {
            "name": candidate.get("name"),
            "state": candidate.get("state"),
            "city": candidate.get("city"),
            "raw_notes": candidate.get("raw_notes"),
            "reasoning": candidate.get("reasoning"),
            "trust_score": candidate.get("trust_score"),
            "trust_flags": candidate.get("trust_flags"),
            "specialties": candidate.get("specialties"),
            "procedures": candidate.get("procedures"),
            "equipment": candidate.get("equipment"),
            "capabilities": candidate.get("capabilities"),
        }

        if distance_info["distance_requested"]:
            facility_payload["lat"] = candidate.get("lat")
            facility_payload["lon"] = candidate.get("lon")

        user_payload = {
            "user_query": user_query,
            "query_plan": query_plan,
            "distance_context": {
                "distance_requested": distance_info["distance_requested"],
                "nearest_only": distance_info["nearest_only"],
                "max_distance_km": distance_info["max_distance_km"],
                "origin_available": distance_info["has_origin"],
            },
            "candidate": facility_payload,
        }

        system_prompt = """
Return JSON ONLY with keys:
match_score,
trust_penalty,
should_recommend,
matched_requirements,
conflicting_flags,
reasoning

Scoring rules:
- Evaluate medical relevance to the user query first.
- Use trust_penalty when candidate evidence is weak, contradictory, vague, or suspicious.
- Only consider distance if distance_context.distance_requested is true.
- If distance is not requested, ignore geographic closeness in scoring.
- Do not invent facts.
- Keep match_score and trust_penalty between 0 and 1.
"""

        response = Agent.serving_endpoints.query(
            name="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
                ChatMessage(role=ChatMessageRole.USER, content=json.dumps(user_payload, ensure_ascii=False)),
            ],
            temperature=0.1,
            max_tokens=500
        )

        content = response.choices[0].message.content
        return parse_json_from_llm_text(content)

    except Exception as e:
        print(f"Warnung: Judge fehlgeschlagen für {candidate.get('name')}: {e}")
        return error_result

In [0]:
@mlflow.trace
def rerank_candidates_with_llm(
    user_query: str,
    query_plan: Dict[str, Any],
    candidates: List[Dict[str, Any]],
    top_k: int = 12
):
    judged = []

    distance_info = should_use_distance(query_plan)

    distance_requested = distance_info["distance_requested"]
    nearest_only = distance_info["nearest_only"]
    max_distance_km = distance_info["max_distance_km"]
    has_origin = distance_info["has_origin"]
    origin_lat = distance_info["origin_lat"]
    origin_lon = distance_info["origin_lon"]

    actual_top_k = min(len(candidates), top_k)

    for candidate in candidates[:actual_top_k]:
        candidate_lat = candidate.get("lat")
        candidate_lon = candidate.get("lon")

        distance_km = None
        distance_score = 0.0

        if distance_requested and has_origin:
            distance_km = haversine_km(origin_lat, origin_lon, candidate_lat, candidate_lon)

            if max_distance_km is not None and distance_km is not None:
                try:
                    if distance_km > float(max_distance_km):
                        continue
                except Exception:
                    pass

            distance_score = distance_to_score(distance_km)

        judgement = llm_judge_candidate(user_query, query_plan, candidate)

        base_trust = float(candidate.get("trust_score") or 0.0)
        match_score = float(judgement.get("match_score", 0.0))
        trust_penalty = float(judgement.get("trust_penalty", 0.0))

        if distance_requested and has_origin and nearest_only:
            distance_weight = 0.20
        elif distance_requested and has_origin and max_distance_km is not None:
            distance_weight = 0.10
        else:
            distance_weight = 0.0

        final_score = (
            (0.55 * match_score)
            + (0.30 * base_trust)
            + (distance_weight * distance_score)
            - (0.25 * trust_penalty)
        )

        final_score = max(0.0, min(1.0, round(final_score, 4)))

        judged.append({
            "name": candidate.get("name"),
            "state": candidate.get("state"),
            "city": candidate.get("city"),
            "final_score": final_score,
            "distance_km": round(distance_km, 2) if distance_km is not None else None,
            "distance_used": bool(distance_requested and has_origin),
            "judge": judgement
        })

    judged.sort(key=lambda x: x["final_score"], reverse=True)
    return judged

In [0]:
@mlflow.trace
def run_facility_search_notebook(
    user_query: str,
    table_name: str = "facilities_scored",
    max_candidates: int = 20,
    top_k: int = 10,
    min_final_score: float = 0.45,
    include_non_recommended: bool = False,
):
    if not user_query or not str(user_query).strip():
        return {
            "ok": False,
            "error": "user_query is empty",
            "query_reasoning": None,
            "results": []
        }

    raw_plan = plan_query(user_query)
    parsed_plan = parse_json_safely(raw_plan)
    query_plan = validate_query_plan(parsed_plan)

    candidate_pool = retrieve_candidate_pool(
        query_plan=query_plan,
        table_name=table_name,
        max_candidates=max_candidates
    )

    ranked_results = rerank_candidates_with_llm(
        user_query=user_query,
        query_plan=query_plan,
        candidates=candidate_pool,
        top_k=top_k
    )

    recommended_results = [
        r for r in ranked_results
        if bool((r.get("judge") or {}).get("should_recommend", False))
        and float(r.get("final_score", 0.0)) >= float(min_final_score)
    ]

    query_reasoning = {
        "original_user_query": user_query,
        "medical_need": query_plan.get("medical_need"),
        "urgency": query_plan.get("urgency"),
        "trust_threshold": query_plan.get("trust_threshold"),
        "distance_preference": query_plan.get("distance_preference"),
        "origin": query_plan.get("origin"),
        "constraints": query_plan.get("constraints", []),
        "reasoning_steps": query_plan.get("reasoning_steps", [])
    }

    results_base = ranked_results if include_non_recommended else recommended_results

    candidate_lookup = {}
    for c in candidate_pool:
        name_key = str(c.get("name") or "").strip().lower()
        if name_key and name_key not in candidate_lookup:
            candidate_lookup[name_key] = c

    results_to_return = []
    for r in results_base:
        name_key = str(r.get("name") or "").strip().lower()
        source_candidate = candidate_lookup.get(name_key, {})

        lat_value = source_candidate.get("lat")
        lon_value = source_candidate.get("lon")

        if lat_value is None:
            lat_value = source_candidate.get("latitude")
        if lon_value is None:
            lon_value = source_candidate.get("longitude")

        results_to_return.append({
            **r,
            "lat": float(lat_value) if lat_value is not None else None,
            "lon": float(lon_value) if lon_value is not None else None,
        })

    return {
        "ok": True,
        "error": None,
        "query_reasoning": query_reasoning,
        "candidate_pool_size": len(candidate_pool),
        "ranked_result_count": len(ranked_results),
        "recommended_result_count": len(recommended_results),
        "results": results_to_return
    }

In [0]:
dbutils.widgets.text("postal_code", "")
postal_code = dbutils.widgets.get("postal_code").strip()

response = run_facility_search_notebook(
    user_query=postal_code,
    table_name="facilities_scored",
    max_candidates=20,
    top_k=10,
    min_final_score=0.45,
    include_non_recommended=False
)

dbutils.notebook.exit(json.dumps(response, ensure_ascii=False))

In [0]:
# 1. UNVERÄNDERT - Funktioniert
import json
import mlflow
import math
from typing import List, Dict, Any

def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))

def compute_desert_scores(
    recommended_results: List[Dict[str, Any]],
    radius_km: float
) -> Dict[str, float]:
    if not recommended_results:
        return {
            "coverage_score": 0.0,
            "desert_score": 1.0,
            "confidence_score": 0.15
        }

    distances = []
    trusts = []

    for r in recommended_results:
        if r.get("distance_km") is not None:
            distances.append(float(r["distance_km"]))
        trusts.append(float(r.get("final_score", 0.0)))

    min_distance = min(distances) if distances else radius_km
    avg_trust = sum(trusts) / len(trusts) if trusts else 0.0
    facility_count_factor = clamp(len(recommended_results) / 5.0)

    distance_factor = 1.0 - clamp(min_distance / radius_km) if radius_km > 0 else 0.0
    trust_factor = clamp(avg_trust)

    coverage_score = (
        0.40 * facility_count_factor +
        0.30 * distance_factor +
        0.30 * trust_factor
    )

    desert_score = 1.0 - coverage_score

    confidence_score = clamp(
        0.5 * facility_count_factor +
        0.5 * trust_factor
    )

    return {
        "coverage_score": round(coverage_score, 4),
        "desert_score": round(desert_score, 4),
        "confidence_score": round(confidence_score, 4)
    }

def build_area_reasoning(
    medical_need: str,
    radius_km: float,
    candidate_count: int,
    recommended_count: int,
    score_bundle: Dict[str, float]
) -> Dict[str, Any]:
    steps = [
        f"Interpreted target capability as '{medical_need}'",
        f"Searched for facilities within {radius_km} km",
        f"Evaluated candidate facilities for recommendation-worthiness",
        f"Computed coverage and desert scores from count, distance, and trust",
        f"Flagged area as desert when coverage is too weak"
    ]

    summary = (
        f"For medical need '{medical_need}', {candidate_count} nearby candidates were checked, "
        f"{recommended_count} were recommendable, resulting in coverage_score="
        f"{score_bundle['coverage_score']} and desert_score={score_bundle['desert_score']}."
    )

    return {
        "summary": summary,
        "reasoning_steps": steps
    }

@mlflow.trace
def detect_medical_deserts_for_points(
    points: List[Dict[str, Any]],
    medical_need: str,
    table_name: str = "facilities_scored",
    search_radius_km: float = 50.0,
    max_candidates: int = 50,
    top_k: int = 20,
    min_final_score: float = 0.45,
    desert_threshold: float = 0.60,
) -> Dict[str, Any]:
    outputs = []

    for point in points:
        lat = float(point["lat"])
        lon = float(point["lon"])
        region_id = point.get("region_id") or f"{lat:.4f}_{lon:.4f}"
        postal_code = point.get("postal_code")

        query_plan = {
            "medical_need": medical_need,
            "urgency": "high",
            "trust_threshold": min_final_score,
            "distance_preference": "nearest",
            "origin": {"lat": lat, "lon": lon},
            "constraints": [
                {"field": "distance_km", "operator": "<=", "value": search_radius_km}
            ],
            "reasoning_steps": [
                "Use geographic origin from region centroid",
                "Find nearby facilities for target capability",
                "Recommend only high-confidence facilities",
                "Estimate whether this region is under-served"
            ]
        }

        candidate_pool = retrieve_candidate_pool(
            query_plan=query_plan,
            table_name=table_name,
            max_candidates=max_candidates
        )

        user_query = f"{medical_need} near lat {lat}, lon {lon}"

        ranked_results = rerank_candidates_with_llm(
            user_query=user_query,
            query_plan=query_plan,
            candidates=candidate_pool,
            top_k=top_k
        )

        recommended_results = [
            r for r in ranked_results
            if bool((r.get("judge") or {}).get("should_recommend", False))
            and float(r.get("final_score", 0.0)) >= float(min_final_score)
        ]

        score_bundle = compute_desert_scores(
            recommended_results=recommended_results,
            radius_km=search_radius_km
        )

        best_distance = None
        if recommended_results:
            dvals = [float(r["distance_km"]) for r in recommended_results if r.get("distance_km") is not None]
            best_distance = min(dvals) if dvals else None

        avg_trust = None
        if recommended_results:
            avg_trust = sum(float(r.get("final_score", 0.0)) for r in recommended_results) / len(recommended_results)

        is_medical_desert = float(score_bundle["desert_score"]) >= float(desert_threshold)

        area_reasoning = build_area_reasoning(
            medical_need=medical_need,
            radius_km=search_radius_km,
            candidate_count=len(candidate_pool),
            recommended_count=len(recommended_results),
            score_bundle=score_bundle
        )

        outputs.append({
            "region_id": region_id,
            "postal_code": postal_code,
            "lat": lat,
            "lon": lon,
            "medical_need": medical_need,
            "search_radius_km": search_radius_km,
            "candidate_pool_size": len(candidate_pool),
            "recommended_facility_count": len(recommended_results),
            "best_facility_distance_km": round(best_distance, 3) if best_distance is not None else None,
            "avg_recommended_trust": round(avg_trust, 4) if avg_trust is not None else None,
            "coverage_score": score_bundle["coverage_score"],
            "desert_score": score_bundle["desert_score"],
            "confidence_score": score_bundle["confidence_score"],
            "is_medical_desert": is_medical_desert,
            "query_reasoning": query_plan,
            "area_reasoning": area_reasoning,
            "top_recommendations": recommended_results[:5]
        })

    outputs = sorted(outputs, key=lambda x: x["desert_score"], reverse=True)

    return {
        "ok": True,
        "medical_need": medical_need,
        "source_table": table_name,
        "search_radius_km": search_radius_km,
        "desert_threshold": desert_threshold,
        "region_count": len(outputs),
        "results": outputs
    }

In [0]:
# 2. GEFIXT - Korrekte Spaltennamen aus Dataset: lat, lon, pincode
import json
from pyspark.sql import functions as F

@mlflow.trace
def process_and_store_medical_deserts(
    medical_need: str,
    source_table: str = "facilities_scored",
    target_table_prefix: str = "medical_desert_results",
    search_radius_km: float = 50.0,
    max_candidates: int = 50,
    top_k: int = 20,
    min_final_score: float = 0.45,
    desert_threshold: float = 0.60,
) -> Dict[str, Any]:

    pin_points_df = (
        spark.table("facilities_scored")
        .filter(F.col("pincode").isNotNull())
        .filter(F.col("lat").isNotNull())
        .filter(F.col("lon").isNotNull())
        .withColumn("postal_code", F.trim(F.col("pincode").cast("string")))
        .withColumn("lat", F.col("lat").cast("double"))
        .withColumn("lon", F.col("lon").cast("double"))
        .groupBy("postal_code")
        .agg(
            F.avg("lat").alias("lat"),
            F.avg("lon").alias("lon"),
            F.count("*").alias("facility_count_in_pin")
        )
        .filter(F.length("postal_code") > 0)
    )

    points = [
        {
            "region_id": row["postal_code"],
            "postal_code": row["postal_code"],
            "lat": float(row["lat"]),
            "lon": float(row["lon"]),
            "facility_count_in_pin": int(row["facility_count_in_pin"])
        }
        for row in pin_points_df.collect()
    ]

    response = detect_medical_deserts_for_points(
        points=points,
        medical_need=medical_need,
        table_name=source_table,
        search_radius_km=search_radius_km,
        max_candidates=max_candidates,
        top_k=top_k,
        min_final_score=min_final_score,
        desert_threshold=desert_threshold
    )

    target_table = f"{target_table_prefix}_{medical_need.replace(' ', '_')}"

    df_final = spark.createDataFrame([{
        "medical_need": medical_need,
        "source_table": source_table,
        "search_radius_km": search_radius_km,
        "max_candidates": max_candidates,
        "top_k": top_k,
        "min_final_score": min_final_score,
        "desert_threshold": desert_threshold,
        "results": json.dumps(response, ensure_ascii=False)
    }])

    (
        df_final.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    return {
        "ok": True,
        "medical_need": medical_need,
        "target_table": target_table,
        "region_count": response.get("region_count", 0)
    }

In [0]:
# 3. UNVERÄNDERT - Funktioniert
result = process_and_store_medical_deserts(
    medical_need="emergency_surgery",
    source_table="facilities_scored",
    target_table_prefix="medical_desert_results",
    search_radius_km=50.0,
    max_candidates=50,
    top_k=20,
    min_final_score=0.45,
    desert_threshold=0.60
)

print(result)

In [0]:
import json

dbutils.widgets.text("medical_need", "emergency_surgery")
medical_need = dbutils.widgets.get("medical_need").strip()

table_name_final = f"medical_desert_results_{medical_need.replace(' ', '_')}"

try:
    rows = spark.sql(f"SELECT results FROM {table_name_final} LIMIT 1").collect()

    if not rows:
        dbutils.notebook.exit(json.dumps({
            "ok": False,
            "error": f"No stored results found for {medical_need}"
        }, ensure_ascii=False))

    result_json = rows[0]["results"]
    dbutils.notebook.exit(result_json)

except Exception as e:
    dbutils.notebook.exit(json.dumps({
        "ok": False,
        "error": str(e)
    }, ensure_ascii=False))